# HIG interactivo 10×10

Pinta una imagen 10×10 con cinco colores y calcula el HIG intrínseco de la segmentación en 4-regiones.

**Uso:** ejecuta las dos celdas. La segunda abrirá una interfaz Gradio. Selecciona un color, pulsa sobre los píxeles y después `Calcular HIG`.

La aplicación calcula los tipos de generadores, las incidencias por bloques, `chi_cell` y `chi_graph`. La identidad `chi_cell = 2` se verifica; no se impone artificialmente.


In [ ]:
!pip -q install -U gradio


In [ ]:
import numpy as np
from collections import defaultdict
import gradio as gr

H=W=10
OUT=-1
COLOR_NAMES=['Rojo','Azul','Verde','Amarillo','Morado']
EMOJI=['🟥','🟦','🟩','🟨','🟪']
PALETTE_OPTIONS=[f'{EMOJI[i]} {COLOR_NAMES[i]}' for i in range(5)]
DIR4=[(-1,0),(1,0),(0,-1),(0,1)]
DIR8=[(dr,dc) for dr in (-1,0,1) for dc in (-1,0,1) if (dr,dc)!=(0,0)]

def label_4_regions(A):
    h,w=A.shape
    rid_map=-np.ones((h,w),dtype=int)
    regions=[]; rid=0
    for r in range(h):
        for c in range(w):
            if rid_map[r,c]!=-1: continue
            color=int(A[r,c]); stack=[(r,c)]; rid_map[r,c]=rid; pixels=set()
            while stack:
                x,y=stack.pop(); pixels.add((x,y))
                for dx,dy in DIR4:
                    nx,ny=x+dx,y+dy
                    if 0<=nx<h and 0<=ny<w and rid_map[nx,ny]==-1 and int(A[nx,ny])==color:
                        rid_map[nx,ny]=rid; stack.append((nx,ny))
            regions.append({'id':rid,'color':color,'pixels':pixels}); rid+=1
    return rid_map,regions

def edge_key(a,b): return tuple(sorted((tuple(a),tuple(b))))

def build_frontier(A,rid_map):
    h,w=A.shape; edge_cells=defaultdict(list)
    for r in range(h):
        for c in range(w):
            for a,b in [((r,c),(r,c+1)),((r+1,c),(r+1,c+1)),((r,c),(r+1,c)),((r,c+1),(r+1,c+1))]:
                edge_cells[edge_key(a,b)].append((r,c))
    frontier=[]; vte=defaultdict(list)
    for key,cells in edge_cells.items():
        if len(cells)==1:
            pair=(OUT,int(rid_map[cells[0]]))
        else:
            r1=int(rid_map[cells[0]]); r2=int(rid_map[cells[1]])
            if r1==r2: continue
            pair=(r1,r2)
        pair=tuple(sorted(pair)); idx=len(frontier)
        frontier.append({'vertices':key,'cells':list(cells),'pair':pair})
        for v in key: vte[v].append(idx)
    return frontier,vte

def frontier_components(frontier,vte):
    edge_comp=[-1]*len(frontier); comps=[]
    for start in range(len(frontier)):
        if edge_comp[start]!=-1: continue
        cid=len(comps); edge_comp[start]=cid; stack=[start]; comp=[]
        while stack:
            e=stack.pop(); comp.append(e)
            for v in frontier[e]['vertices']:
                for f in vte[v]:
                    if edge_comp[f]==-1:
                        edge_comp[f]=cid; stack.append(f)
        comps.append(comp)
    return comps,edge_comp

def continuation_classes(frontier,vte):
    junctions={v for v,edges in vte.items() if len(edges)>=3}
    cls=[-1]*len(frontier); classes=[]
    for start in range(len(frontier)):
        if cls[start]!=-1: continue
        cid=len(classes); cls[start]=cid; pair=frontier[start]['pair']; stack=[start]; support=[]
        while stack:
            e=stack.pop(); support.append(e)
            for v in frontier[e]['vertices']:
                if v in junctions: continue
                for f in vte[v]:
                    if cls[f]==-1 and frontier[f]['pair']==pair:
                        cls[f]=cid; stack.append(f)
        ends=[]
        for e in support:
            for v in frontier[e]['vertices']:
                if v in junctions: ends.append(v)
        if len(ends)==0: typ='enc0'
        elif len(ends)==2: typ='enc1' if ends[0]==ends[1] else 'seg'
        else: raise RuntimeError(f'Clase de frontera inválida: {len(ends)} extremos de junction.')
        classes.append({'id':f'G{cid}','type':typ,'edges':support,'pair':pair})
    return classes,sorted(junctions)

def holes_of_region(rid,rid_map):
    h,w=rid_map.shape; seen=np.zeros((h,w),dtype=bool); holes=[]
    for r in range(h):
        for c in range(w):
            if rid_map[r,c]==rid or seen[r,c]: continue
            stack=[(r,c)]; seen[r,c]=True; comp=set(); touches=False
            while stack:
                x,y=stack.pop(); comp.add((x,y)); touches |= (x in (0,h-1) or y in (0,w-1))
                for dx,dy in DIR8:
                    nx,ny=x+dx,y+dy
                    if 0<=nx<h and 0<=ny<w and not seen[nx,ny] and rid_map[nx,ny]!=rid:
                        seen[nx,ny]=True; stack.append((nx,ny))
            if not touches: holes.append(comp)
    return holes

def identify_cuts(regions,rid_map,frontier,components,edge_comp):
    frame_free=[cid for cid,comp in enumerate(components) if all(OUT not in frontier[e]['pair'] for e in comp)]
    cut_map={}
    for reg in regions:
        rid=reg['id']
        for hole in holes_of_region(rid,rid_map):
            interface=[]
            for ei,e in enumerate(frontier):
                cells=e['cells']
                if len(cells)!=2: continue
                p,q=cells
                if (p in hole and int(rid_map[q])==rid) or (q in hole and int(rid_map[p])==rid): interface.append(ei)
            if not interface: raise RuntimeError('Agujero sin interfaz de frontera.')
            cids={edge_comp[e] for e in interface}
            if len(cids)!=1: raise RuntimeError('La interfaz de un agujero cae en varias componentes de F.')
            cid=next(iter(cids))
            if cid in cut_map and cut_map[cid][0]!=rid: raise RuntimeError('Una componente cut recibió dos carriers distintos.')
            cut_map[cid]=(rid,hole)
    if set(frame_free)!=set(cut_map):
        raise RuntimeError('Falló la comprobación cut-hole: componentes frame-free y agujeros no están en biyección.')
    return [{'id':f'C{cid}','type':'cut','component':cid,'edges':components[cid], 'carrier':cut_map[cid][0], 'hole':cut_map[cid][1]} for cid in frame_free]

def compute_hig(grid):
    A=np.array(grid,dtype=int)
    rid_map,regions=label_4_regions(A)
    frontier,vte=build_frontier(A,rid_map)
    comps,edge_comp=frontier_components(frontier,vte)
    cont,junctions=continuation_classes(frontier,vte)
    cuts=identify_cuts(regions,rid_map,frontier,comps,edge_comp)
    seg=[g for g in cont if g['type']=='seg']; enc1=[g for g in cont if g['type']=='enc1']; enc0=[g for g in cont if g['type']=='enc0']
    V1=seg+enc1+enc0+cuts; V1_types=['seg']*len(seg)+['enc1']*len(enc1)+['enc0']*len(enc0)+['cut']*len(cuts)
    V0=[('junc',v) for v in junctions]+[('enc',g['id']) for g in enc0]; V0_types=['junc']*len(junctions)+['enc']*len(enc0)
    V2=list(range(len(regions)))+[OUT]; V2_types=['img']*len(regions)+['OUT']
    jr={v:i for i,v in enumerate(junctions)}; er={g['id']:len(junctions)+i for i,g in enumerate(enc0)}; c2={rid:i for i,rid in enumerate(V2)}
    M01=np.zeros((len(V0),len(V1)),dtype=int); M12=np.zeros((len(V1),len(V2)),dtype=int)
    for j,g in enumerate(V1):
        branches=defaultdict(int)
        for ei in g['edges']:
            for v in frontier[ei]['vertices']:
                if v in jr: branches[v]+=1
        for v,m in branches.items(): M01[jr[v],j]=m
        if g['type']=='enc0': M01[er[g['id']],j]=2
        if g['type']=='cut': M12[j,c2[g['carrier']]]=2
        else:
            a,b=g['pair']; M12[j,c2[a]]=1; M12[j,c2[b]]=1
    n0,n1,n2=len(V0),len(V1),len(V2); chi_cell=n2-n1+n0
    if chi_cell!=2: raise AssertionError(f'χcell={chi_cell}; la implementación es inconsistente para esta imagen.')
    V=n0+n1+n2; E=int(M01.sum()+M12.sum()); chi_graph=V-E
    regions_by_color=[sum(1 for R in regions if R['color']==i) for i in range(5)]
    return {'regions':regions,'regions_by_color':regions_by_color,'junctions':junctions,'seg':seg,'enc1':enc1,'enc0':enc0,'cuts':cuts,'V0':V0,'V1':V1,'V2':V2,'V0_types':V0_types,'V1_types':V1_types,'V2_types':V2_types,'M01':M01,'M12':M12,'chi_cell':chi_cell,'graph_vertices':V,'graph_edges':E,'chi_graph':chi_graph}

def block_stats(M,rows,cols,rt,ct):
    ri=[i for i,t in enumerate(rows) if t==rt]; ci=[j for j,t in enumerate(cols) if t==ct]
    if not ri or not ci: return (0,0)
    B=M[np.ix_(ri,ci)]; return int(np.count_nonzero(B)),int(B.sum())

def interaction_rows(R):
    out=[]
    for a,b,ta,tb in [('V0^junc','V1^seg','junc','seg'),('V0^junc','V1^enc,1','junc','enc1'),('V0^junc','V1^enc,0','junc','enc0'),('V0^junc','V1^cut','junc','cut'),('V0^enc','V1^enc,0','enc','enc0')]:
        out.append((a,b,*block_stats(R['M01'],R['V0_types'],R['V1_types'],ta,tb)))
    for a,b,ta,tb in [('V1^seg','V2^img','seg','img'),('V1^seg','OUT','seg','OUT'),('V1^enc,1','V2^img','enc1','img'),('V1^enc,1','OUT','enc1','OUT'),('V1^enc,0','V2^img','enc0','img'),('V1^enc,0','OUT','enc0','OUT'),('V1^cut','V2^img','cut','img')]:
        out.append((a,b,*block_stats(R['M12'],R['V1_types'],R['V2_types'],ta,tb)))
    return out

def report_html(grid):
    R=compute_hig(grid)
    n2=len(R['V2']); n1=len(R['V1']); n0=len(R['V0'])
    rows=''.join(f'<tr><td>{a}</td><td>{b}</td><td>{nz}</td><td><b>{mult}</b></td></tr>' for a,b,nz,mult in interaction_rows(R))
    colors=''.join(f'<tr><td>{EMOJI[i]} {COLOR_NAMES[i]}</td><td>{R["regions_by_color"][i]}</td></tr>' for i in range(5))
    return f'''<div class="report"><h3>HIG</h3><table><tr><th>Generador</th><th>N</th></tr><tr><td>V2^img</td><td>{len(R['regions'])}</td></tr><tr><td>OUT</td><td>1</td></tr><tr><td><b>V2</b></td><td><b>{n2}</b></td></tr><tr><td>V1^seg</td><td>{len(R['seg'])}</td></tr><tr><td>V1^enc,1</td><td>{len(R['enc1'])}</td></tr><tr><td>V1^enc,0</td><td>{len(R['enc0'])}</td></tr><tr><td>V1^cut</td><td>{len(R['cuts'])}</td></tr><tr><td><b>V1</b></td><td><b>{n1}</b></td></tr><tr><td>V0^junc</td><td>{len(R['junctions'])}</td></tr><tr><td>V0^enc</td><td>{len(R['enc0'])}</td></tr><tr><td><b>V0</b></td><td><b>{n0}</b></td></tr></table><h4>Euler</h4><p><b>χcell = {n2} − {n1} + {n0} = {R['chi_cell']} ✓</b><br>|V(HIG)|={R['graph_vertices']}<br>|E(HIG)|={R['graph_edges']} (con multiplicidad)<br><b>χgraph={R['chi_graph']}</b></p><h4>4-regiones por color</h4><table><tr><th>Color</th><th>Regiones</th></tr>{colors}</table><h4>Interacciones</h4><table><tr><th>Tipo A</th><th>Tipo B</th><th>nnz</th><th>multiplicidad</th></tr>{rows}</table><p>HIGM01: {R['M01'].shape}, nnz={np.count_nonzero(R['M01'])}, Σ={R['M01'].sum()}<br>HIGM12: {R['M12'].shape}, nnz={np.count_nonzero(R['M12'])}, Σ={R['M12'].sum()}</p></div>'''

# Callbacks UI

def paint(grid,palette,r,c):
    g=np.array(grid,dtype=int)
    idx=PALETTE_OPTIONS.index(palette)
    g[r,c]=idx
    return g.tolist(), gr.update(value=EMOJI[idx])

def reset():
    g=np.zeros((H,W),dtype=int).tolist()
    return [g]+[gr.update(value=EMOJI[0]) for _ in range(H*W)]+[report_html(g)]

def randomize():
    g=np.random.default_rng().integers(0,5,size=(H,W)).tolist()
    updates=[gr.update(value=EMOJI[g[r][c]]) for r in range(H) for c in range(W)]
    return [g]+updates+[report_html(g)]

def calc(grid):
    try: return report_html(grid)
    except Exception as e: return f'<div style="color:#b00020"><b>Error:</b> {e}</div>'

CSS='''
.pixel button, button.pixel {min-width:36px !important; width:36px !important; height:36px !important; padding:0 !important; font-size:22px !important;}
.report table {border-collapse:collapse; width:100%; font-size:13px}.report th,.report td{border:1px solid #bbb;padding:4px}.report h3,.report h4{margin:8px 0 4px}
'''
with gr.Blocks(title='HIG 10x10') as demo:
    state=gr.State(np.zeros((H,W),dtype=int).tolist())
    gr.Markdown('# HIG interactivo 10×10')
    with gr.Row():
        with gr.Column(scale=1):
            palette=gr.Radio(PALETTE_OPTIONS,value=PALETTE_OPTIONS[0],label='Color activo')
            pixels=[]
            for r in range(H):
                with gr.Row():
                    for c in range(W):
                        b=gr.Button(EMOJI[0],elem_classes=['pixel'],min_width=36)
                        pixels.append(b)
                        b.click(fn=lambda g,p,rr=r,cc=c: paint(g,p,rr,cc),inputs=[state,palette],outputs=[state,b],show_progress='hidden')
            with gr.Row():
                calc_btn=gr.Button('Calcular HIG',variant='primary')
                reset_btn=gr.Button('Limpiar')
                rnd_btn=gr.Button('Aleatoria')
        with gr.Column(scale=2):
            report=gr.HTML(report_html(np.zeros((H,W),dtype=int).tolist()))
    calc_btn.click(calc,inputs=state,outputs=report)
    reset_btn.click(reset,inputs=None,outputs=[state]+pixels+[report])
    rnd_btn.click(randomize,inputs=None,outputs=[state]+pixels+[report])

demo.launch(inline=True, share=True, debug=True, css=CSS)
